In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder, OneHotEncoder
import pickle

In [3]:
data = pd.read_csv("./Churn_Modelling.csv")

data

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9996,15606229,Obijiaku,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,9997,15569892,Johnstone,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,9998,15584532,Liu,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,9999,15682355,Sabbatini,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


In [4]:
# Preprocess the data
data = data.drop(["RowNumber","CustomerId","Surname"],axis=1)

In [5]:
# Encode categorical variables
label_encoder_gender = LabelEncoder()
data["Gender"] = label_encoder_gender.fit_transform(data["Gender"])

In [9]:
# One Hot Encode "Geography"
onehot_encoder_geo = OneHotEncoder(handle_unknown="ignore")
geo_encoded= onehot_encoder_geo.fit_transform(data[["Geography"]]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded,columns=onehot_encoder_geo.get_feature_names_out(["Geography"]))
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [10]:
# Combine one-hot encoded columns with original data
data = pd.concat([data.drop("Geography",axis=1),geo_encoded_df],axis=1)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [11]:
# Split the data into features  and target
X = data.drop("EstimatedSalary",axis=1)
y = data["EstimatedSalary"]

In [12]:
## Split the data in training and testing sets
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [13]:
## Scale these features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [14]:
## Save the encoders and scaler
with open("label_encoder_gender.pkl","wb") as file:
    pickle.dump(label_encoder_gender,file)

with open("onehot_encoder_geo.pkl","wb") as file:
    pickle.dump(onehot_encoder_geo,file)

with open("scaler.pkl","wb") as file:
    pickle.dump(scaler,file)

## ANN Regression Problem Statem

In [15]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [16]:
## Build Our ANN Model
model = Sequential([
    Dense(64,activation="relu",input_shape=(X_train.shape[1],)), ## HL1 connected with input layer
    Dense(32,activation="relu"), ##L2
    Dense(1) ## output layer
])

## Compile the model
model.compile(optimizer="adam",loss="mean_absolute_error",metrics=["mae"])

In [17]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                832       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [19]:
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

log_dir = "regressionlogs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback = TensorBoard(log_dir=log_dir,histogram_freq=1)

In [20]:
early_stopping_callback = EarlyStopping(monitor="val_loss",patience=10,restore_best_weights=True)

In [21]:
### Train the model
history = model.fit(
    X_train,y_train,validation_data =(X_test,y_test),epochs=100,callbacks=[early_stopping_callback,tensorflow_callback]
)

Epoch 1/100
250/250 [==============================] - 1s 3ms/step - loss: 100378.6797 - mae: 100378.6797 - val_loss: 98525.1016 - val_mae: 98525.1016
Epoch 2/100
250/250 [==============================] - 0s 2ms/step - loss: 99660.8594 - mae: 99660.8594 - val_loss: 97071.2422 - val_mae: 97071.2422
Epoch 3/100
250/250 [==============================] - 0s 1ms/step - loss: 97118.8047 - mae: 97118.8047 - val_loss: 93326.8438 - val_mae: 93326.8438
Epoch 4/100
250/250 [==============================] - 0s 1ms/step - loss: 92083.3359 - mae: 92083.3359 - val_loss: 87043.3281 - val_mae: 87043.3281
Epoch 5/100
250/250 [==============================] - 0s 2ms/step - loss: 84685.6016 - mae: 84685.6016 - val_loss: 78830.6406 - val_mae: 78830.6406
Epoch 6/100
250/250 [==============================] - 0s 2ms/step - loss: 75833.3125 - mae: 75833.3125 - val_loss: 70075.4766 - val_mae: 70075.4766
Epoch 7/100
250/250 [==============================] - 0s 2ms/step - loss: 67039.6719 - mae: 67039.6719 

In [ ]:
## Evaluate the model

In [22]:
%load_ext tensorboard

In [25]:
%tensorboard --logdir regressionlogs/fit

Reusing TensorBoard on port 6008 (pid 127275), started 0:01:11 ago. (Use '!kill 127275' to kill it.)

In [27]:
## Evaluate model on the test data
test_loss,test_mae = model.evaluate(X_test,y_test)
print(f'Test MAE : {test_mae}')

63/63 [==============================] - 0s 811us/step - loss: 50351.1719 - mae: 50351.1719
Test MAE : 50351.171875


In [28]:
model.save("regression_model.h5")

/workspaces/ann-classification/venv/lib/python3.11/site-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
